## Setup


In [3]:
# Install packages
# %pip install -r requirements.txt

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import math
import statistics as stats
import scipy

In [5]:
def code_break():
    print("-----------------------------------------")

def get_ssr(matrix: np.ndarray, estimators: np.array):
    ""

In [6]:
apple = pd.read_csv("APPLE.csv")
apple.head()

,id,educ,date,state,regprc,ecoprc,inseason,hhsize,male,faminc,age,reglbs,ecolbs,numlt5,num5_17,num18_64,numgt64
0,10002,16,111597,SD,1.19,1.19,1,4,0,45,43,2.0,2.000000,0,1,3,0
1,10004,16,121897,KS,0.59,0.79,0,1,0,65,37,0.0,2.000000,0,0,1,0
2,10034,18,111097,MI,0.59,0.99,1,3,0,65,44,0.0,2.666667,0,2,1,0
3,10035,12,111597,TN,0.89,1.09,1,2,1,55,55,3.0,0.000000,0,0,2,0
4,10039,15,122997,NY,0.89,1.09,0,1,1,25,22,0.0,3.000000,0,0,1,0


## Question 3 ? Manual coefficient estimates


## Estimators

In [7]:
# 3. 

X = apple[['regprc', 'ecoprc', 'inseason']]
X = sm.add_constant(X)
y = apple['ecolbs']

# ----- Manual Calc ------
X = np.array(X)
XtX = X.T @ X

print(f"XtX: {XtX}")

det_XtX = np.linalg.det(XtX)
# display(f"Determinant XtX: {det_XtX}")
inv_XtX = np.linalg.inv(XtX)

print(f"inv_XtX: {inv_XtX}")

# code_break()

Xty = X.T @ y
# print(Xty)

# code_break()

estimators = inv_XtX @ Xty
# print(estimators)

# estimators_results = zip(['const', 'x1', 'x2', 'x3'], estimators)
estimators_results = pd.DataFrame(estimators, index = ['const', 'x1', 'x2', 'x3'])
estimators_results.rename(mapper={0:'manual'}, axis=1, inplace=True)
estimators_results

XtX: [[660.    582.6   713.8   222.   ]
 [582.6   553.662 669.65  192.78 ]
 [713.8   669.65  829.558 235.38 ]
 [222.    192.78  235.38  222.   ]]
inv_XtX: [[ 2.46539584e-02 -1.14262776e-02 -1.11706080e-02 -2.88776250e-03]
 [-1.14262776e-02  8.19466817e-02 -5.63098238e-02 -3.08169376e-05]
 [-1.11706080e-02 -5.63098238e-02  5.61084858e-02  5.78681982e-04]
 [-2.88776250e-03 -3.08169376e-05  5.78681982e-04  6.80546847e-03]]


,manual
const,2.032336
x1,3.029629
x2,-2.939899
x3,-0.157975


## Question 4 ? Homoskedastic covariance and standard errors


## Standard Errors

In [8]:
# Manual Calculation
beta = np.asarray(estimators).reshape(-1)
e = np.asarray(y).reshape(-1) - X @ beta
e_2 = e**2
ssr = e_2.sum()

s_2 = ssr / (X.shape[0] - X.shape[1])

cov_matrix = s_2*(inv_XtX)
print(cov_matrix)

[[ 1.52109803e-01 -7.04977595e-02 -6.89203310e-02 -1.78168948e-02]
 [-7.04977595e-02  5.05594006e-01 -3.47419917e-01 -1.90134104e-04]
 [-6.89203310e-02 -3.47419917e-01  3.46177704e-01  3.57034764e-03]
 [-1.78168948e-02 -1.90134104e-04  3.57034764e-03  4.19883269e-02]]


In [9]:
manual_s2 = cov_matrix.diagonal()


manual_s2_table = pd.DataFrame(
    manual_s2, 
    index=['const', 'regprc', 'ecoprc', 'inseason'],
    columns=['manual']
)
manual_s2_table

,manual
const,0.152110
regprc,0.505594
ecoprc,0.346178
inseason,0.041988


## Question 5 ? Compare with regression output


In [10]:
model1 = sm.OLS(exog=X, endog=y)
results = model1.fit()

results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 ecolbs   R-squared:                       0.037
Model:                            OLS   Adj. R-squared:                  0.033
Method:                 Least Squares   F-statistic:                     8.469
Date:                Fri, 20 Nov 2026   Prob (F-statistic):           1.58e-05
Time:                        18:48:36   Log-Likelihood:                -1535.0
No. Observations:                 660   AIC:                             3078.
Df Residuals:                     656   BIC:                             3096.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.0323      0.390      5.211      0.000       1.267       2.798
x1             3.0296      0.711      4.261      0.000       1.633       4.426
x2            -2.9399      0.588     -4.997      0.000      -4.095      -1.785
x3            -0.1580      0.205     -0.771      0.441      -0.560       0.244
==============================================================================
Omnibus:                      989.100   Durbin-Watson:                   2.025
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           351898.328
Skew:                           8.279   Prob(JB):                         0.00
Kurtosis:                     114.902   Cond. No.                         16.3
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [11]:
stats_package_est = pd.DataFrame(results.params)
# comparison_table.reset_index(inplace=True)
stats_package_est.rename(mapper={0:'stats_package'}, inplace=True, axis=1)
stats_package_est

comparison_table = pd.merge(stats_package_est, estimators_results, left_index=True, right_index=True)

display(comparison_table)

,stats_package,manual
const,2.032336,2.032336
x1,3.029629,3.029629
x2,-2.939899,-2.939899
x3,-0.157975,-0.157975


In [12]:
# Compare s2
stats_package_s2 = np.asarray(results.cov_params()).diagonal()
stats_package_s2_table = pd.DataFrame(stats_package_s2, index=['const', 'regprc', 'ecoprc', 'inseason'], columns=['stats_package'])

pd.merge(manual_s2_table, stats_package_s2_table, left_index=True, right_index=True)

,manual,stats_package
const,0.152110,0.152110
regprc,0.505594,0.505594
ecoprc,0.346178,0.346178
inseason,0.041988,0.041988


## Question 6 ? FWL theorem


## Question 7 ? Model (2) and coefficient interpretation


In [13]:
X = apple[['ecoprc', 'regprc', 'reglbs', 'age', 'faminc', 'hhsize', 'numlt5']]
X = sm.add_constant(X)
y = apple['ecolbs']

In [14]:
model2 = sm.OLS(exog=X, endog=y)
results2 = model2.fit()

results2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 ecolbs   R-squared:                       0.066
Model:                            OLS   Adj. R-squared:                  0.056
Method:                 Least Squares   F-statistic:                     6.583
Date:                Fri, 20 Nov 2026   Prob (F-statistic):           1.57e-07
Time:                        18:48:36   Log-Likelihood:                -1525.0
No. Observations:                 660   AIC:                             3066.
Df Residuals:                     652   BIC:                             3102.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.7147      0.561      3.056      0.002       0.613       2.817
ecoprc        -2.7526      0.583     -4.722      0.000      -3.897      -1.608
regprc         2.7797      0.705      3.942      0.000       1.395       4.164
reglbs        -0.1386      0.033     -4.192      0.000      -0.203      -0.074
age            0.0005      0.007      0.075      0.940      -0.013       0.014
faminc         0.0031      0.003      1.156      0.248      -0.002       0.008
hhsize         0.1119      0.074      1.509      0.132      -0.034       0.257
numlt5        -0.2027      0.176     -1.155      0.249      -0.547       0.142
==============================================================================
Omnibus:                     1009.586   Durbin-Watson:                   2.018
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           391896.643
Skew:                           8.594   Prob(JB):                         0.00
Kurtosis:                     121.133   Cond. No.                         694.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

## Question 8 ? Own-price hypothesis test and confidence interval


In [15]:
coefficient = results2.params.loc['ecoprc']
standard_error = results2.cov_params().loc['ecoprc', 'ecoprc']

# Ho: B >= 0, Ha: B < 0; a = 0.05
a = 0.05
test = coefficient / standard_error
critical_value = stats.NormalDist().inv_cdf(a)
if critical_value > test: 
    print(f"Reject the null since {round(test, 5)} < {round(critical_value, 5)}")
else: 
    print("fail to reject")

# Confidence Interval
code_break()
critical_value = -1*stats.NormalDist().inv_cdf(a/2)
# print(critical_value)

lower_bound = coefficient - (critical_value*standard_error)
upper_bound = coefficient + (critical_value*standard_error)

print("CI: ", lower_bound, ",", upper_bound)

Reject the null since -8.09934 < -1.64485
-----------------------------------------
CI:  -3.4186857601763174 , -2.0864854255491734


## Question 9 ? HC2 covariance and revised inference


## HC2 Errors

In [16]:
## Diagonal
X_np = np.asarray(X)
inv_XtX = np.linalg.inv(X_np.T @ X_np)
projection_matrix = X_np @ inv_XtX @ X_np.T
# print(projection_matrix)
# code_break()
# print(projection_matrix.shape)
# code_break()
px_diag = projection_matrix.diagonal()

## residuals
u2 = np.asarray((results2.resid)**2)

print(type(px_diag),type(u2))
print(px_diag.shape, u2.shape)

## Omega
omega_weights = np.asarray(results2.resid)**2 / (1 - px_diag)

## HC2 Errors
XtX = X.T @ X
inv_XtX = np.linalg.inv(XtX)

hc2_cov = inv_XtX @ (X_np.T * omega_weights) @ X_np @ inv_XtX
code_break()
pd.DataFrame(hc2_cov, columns=model2.exog_names, index=model2.exog_names)

<class 'numpy.ndarray'> <class 'numpy.ndarray'>
(660,) (660,)
-----------------------------------------


,const,ecoprc,regprc,reglbs,age,faminc,hhsize,numlt5
const,0.212772,-0.136302,0.156867,-2.536973e-03,-0.002762,-2.742463e-04,-0.013898,-0.017852
ecoprc,-0.136302,0.317567,-0.465830,-3.140758e-03,0.002669,5.515218e-05,0.017157,0.010413
regprc,0.156867,-0.465830,0.844100,3.079773e-03,-0.005506,2.477389e-04,-0.037203,-0.020476
reglbs,-0.002537,-0.003141,0.003080,2.087790e-03,0.000018,-5.829303e-07,-0.000229,0.000625
age,-0.002762,0.002669,-0.005506,1.790747e-05,0.000076,-2.644274e-06,0.000319,0.000489
faminc,-0.000274,0.000055,0.000248,-5.829303e-07,-0.000003,4.671345e-06,-0.000034,-0.000005
hhsize,-0.013898,0.017157,-0.037203,-2.288426e-04,0.000319,-3.394481e-05,0.005283,-0.003016
numlt5,-0.017852,0.010413,-0.020476,6.245691e-04,0.000489,-4.607283e-06,-0.003016,0.022393


In [17]:
pd.DataFrame(results2.cov_HC2, columns=model2.exog_names, index=model2.exog_names)

,const,ecoprc,regprc,reglbs,age,faminc,hhsize,numlt5
const,0.212772,-0.136302,0.156867,-2.536973e-03,-0.002762,-2.742463e-04,-0.013898,-0.017852
ecoprc,-0.136302,0.317567,-0.465830,-3.140758e-03,0.002669,5.515218e-05,0.017157,0.010413
regprc,0.156867,-0.465830,0.844100,3.079773e-03,-0.005506,2.477389e-04,-0.037203,-0.020476
reglbs,-0.002537,-0.003141,0.003080,2.087790e-03,0.000018,-5.829303e-07,-0.000229,0.000625
age,-0.002762,0.002669,-0.005506,1.790747e-05,0.000076,-2.644274e-06,0.000319,0.000489
faminc,-0.000274,0.000055,0.000248,-5.829303e-07,-0.000003,4.671345e-06,-0.000034,-0.000005
hhsize,-0.013898,0.017157,-0.037203,-2.288426e-04,0.000319,-3.394481e-05,0.005283,-0.003016
numlt5,-0.017852,0.010413,-0.020476,6.245691e-04,0.000489,-4.607283e-06,-0.003016,0.022393


## Question 10 ? Joint F test for the D variables


In [ ]:
X_res_cols = X.columns.to_list()[-5:-1]
print(X_res_cols)
X_RES = apple[X_res_cols]
X_RES = sm.add_constant(X_RES)

model_res = sm.OLS(exog=X_RES, endog=y)
results_res = model_res.fit()

# print(results_res.summary())
# code_break()
# print(results2.summary())

# Get SSE for Restricted and Unrestricted
sse_res = sum((y - results_res.predict(X_RES))**2)
sse_unres = sum((y - results2.predict(X))**2)
code_break()
print({
    'ur':sse_res,
    'r':sse_unres
})

num_df = len(X_res_cols)
den_df = X_RES.shape[0] - len(results2.params)

# F-stat
f_stat = ((sse_res - sse_unres) / num_df) / (sse_unres / (den_df))
code_break()
print(f"F-stat: {f_stat}")

print(f"P-value: {scipy.stats.f.sf(f_stat, num_df, den_df)}")
print(f"Crit. Value: {scipy.stats.f.ppf(0.95, num_df, den_df)}")

['reglbs', 'age', 'faminc', 'hhsize']
-----------------------------------------
{'ur': 4072.4286118615814, 'r': 3926.629707325432}
-----------------------------------------
F-stat: 6.052320491299836
P-value: 8.731501536944279e-05
Crit. Value: 2.385595844962613
